In [1]:
# Cell 0 — Bootstrap: locate project root via .env, then import shared config + download-log helpers
import sys
from pathlib import Path

# Walk up from the current dir until we find the folder containing .env, then add its src/ to sys.path
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *                                              # PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, DOWNLOADS_DIR, SSL_VERIFY, BROWSER_HEADERS, ...
from download_log import load_log, update_entry, print_entry, print_stale_sources  # download-log helpers

log = load_log()   # load the download-log registry so update_entry()/print_entry() work later in the notebook

### FATF — Mutual Evaluation Ratings (AML/CFT, Concept 9) (MANUAL — Cloudflare-gated)

The fatf-gafi.org site is behind Cloudflare's anti-bot challenge, so the files **cannot** be fetched by `curl`/`requests`/the pipeline — they must be downloaded by a real browser. A scripted download returns a ~5 KB "Just a moment..." HTML stub instead of the file.

1. In a browser, go to the FATF Consolidated Assessment Ratings page: https://www.fatf-gafi.org/en/publications/Mutualevaluations/Assessment-ratings.html
2. Download **both** Excel files (click the "Download" button under each):
   - *Consolidated assessment ratings under the 2022 Methodology – Excel*
   - *Consolidated assessment ratings under the 2013 Methodology – Excel*
3. Leave both files in `~/Downloads` with their default names (`consolidated-assessment-ratings-2022-methodology.xlsx` and `consolidated-assessment-ratings-2013-methodology.xlsx`). Verify they are real spreadsheets, not ~5 KB HTML stubs: `ls -la ~/Downloads/consolidated-assessment-ratings-*.xlsx` (the 2013 file is ~150 KB, the 2022 file ~16 KB; a few-KB file means Cloudflare blocked the download — retry in the browser).
4. Re-run `notebooks/exploration/35_fatf_pipeline.ipynb` — both files are auto-detected by glob, the methodology round is parsed from each filename, and the data as-of date is derived from the latest report date in the data. **No manual edit of the notebook is needed**, even when a new methodology round appears (a future `*-2030-methodology.xlsx` is picked up automatically).

*Dependency: requires `pycountry` (in `environment.yml`). New rounds: if FATF introduces a third methodology file, it flows through with no code change; only if they change the in-sheet column layout (banner rows, column order) would the notebook's positional column-mapping need updating — it would raise a clear error in that case.*

In [2]:
# Cell 2 — Discover FATF rating files in Downloads and parse each one's methodology round from its filename.
# Manual-download source (Cloudflare-gated, no auto-fetch). The methodology round is NOT in the sheet content —
# the filename is the only place it appears — so we glob the stable stem and extract the round token per file.
# This means new rounds (e.g. a future *-2030-methodology.xlsx) are picked up automatically with no code change.
import re

_downloads = Path(DOWNLOADS_DIR)   # DOWNLOADS_DIR is a str in config; wrap for globbing

# Glob the version-agnostic stem: match any "...-<something>-methodology....xlsx"
_fatf_files = sorted(_downloads.glob("consolidated-assessment-ratings-*methodology*.xlsx"))
assert _fatf_files, f"No FATF ratings files found in {_downloads} (expected consolidated-assessment-ratings-*methodology*.xlsx)"

# Parse the round token out of each filename (the segment before '-methodology'); fail loudly if a file doesn't match.
fatf_sources = {}   # {round_token: Path}
for p in _fatf_files:
    m = re.search(r"consolidated-assessment-ratings-(.+?)-methodology", p.name)
    assert m, f"Could not parse methodology round from filename: {p.name}"
    fatf_sources[m.group(1)] = p

# Report what was discovered (count and sizes derived from the files themselves — nothing hardcoded).
print(f"Found {len(fatf_sources)} FATF ratings file(s):")
for rnd, p in sorted(fatf_sources.items()):
    print(f"  round={rnd!r}: {p.name} ({p.stat().st_size:,} bytes)")

Found 2 FATF ratings file(s):
  round='2013': consolidated-assessment-ratings-2013-methodology.xlsx (156,428 bytes)
  round='2022': consolidated-assessment-ratings-2022-methodology.xlsx (16,724 bytes)


In [3]:
# Cell 3 — Load each discovered FATF file, tag rows with their methodology round, concatenate into one frame.
# Header is on the 4th row (rows 0-2 are banner/description rows), so skiprows=3.
# We loop over fatf_sources (discovered in Cell 2) — no per-round branching, so new rounds flow through unchanged.
import pandas as pd

# The 56 real columns, in sheet order: 4 ID columns, 11 Immediate Outcomes, 40 Recommendations, 2 change-counts.
# Defined by position (not by reading the messy header text) because the header labels contain newlines/long text.
io_cols = [f"IO{i}" for i in range(1, 12)]        # IO1..IO11  (effectiveness)
rec_cols = [f"R{i}" for i in range(1, 41)]         # R1..R40    (technical compliance)
id_cols = ["jurisdiction", "report_type", "report_date", "assessment_body"]
keep_cols = id_cols + io_cols + rec_cols + ["n_upgrades", "n_downgrades"]   # 57 columns total

frames = []
for rnd, path in sorted(fatf_sources.items()):
    # Read the 'Ratings only' sheet; skip the 3 banner rows so row 4 becomes the header, then we overwrite names by position.
    raw = pd.read_excel(path, sheet_name="Ratings only", skiprows=3, header=0, dtype=str)
    # Keep only the first 57 columns (the rest are empty padding out to ~1549 cols) and assign clean positional names.
    raw = raw.iloc[:, :len(keep_cols)].copy()
    raw.columns = keep_cols
    # Drop fully-empty rows (trailing blanks / spacer rows) — a row with no jurisdiction is not data.
    raw = raw[raw["jurisdiction"].notna() & (raw["jurisdiction"].str.strip() != "")].copy()
    # Tag every row with the methodology round parsed from its filename (the source of truth for the round flag).
    raw["methodology_round"] = rnd
    frames.append(raw)
    print(f"round={rnd!r}: loaded {len(raw)} data rows")

# Concatenate all rounds into one long frame.
fatf_raw = pd.concat(frames, ignore_index=True)
print(f"\nCombined: {len(fatf_raw)} rows, {fatf_raw.shape[1]} columns")
print("Columns:", list(fatf_raw.columns))

round='2013': loaded 692 data rows
round='2022': loaded 7 data rows

Combined: 699 rows, 58 columns
Columns: ['jurisdiction', 'report_type', 'report_date', 'assessment_body', 'IO1', 'IO2', 'IO3', 'IO4', 'IO5', 'IO6', 'IO7', 'IO8', 'IO9', 'IO10', 'IO11', 'R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R11', 'R12', 'R13', 'R14', 'R15', 'R16', 'R17', 'R18', 'R19', 'R20', 'R21', 'R22', 'R23', 'R24', 'R25', 'R26', 'R27', 'R28', 'R29', 'R30', 'R31', 'R32', 'R33', 'R34', 'R35', 'R36', 'R37', 'R38', 'R39', 'R40', 'n_upgrades', 'n_downgrades', 'methodology_round']


In [4]:
# Cell 4 — Collapse to one row per country: latest assessment, newer methodology round wins, flag retained.
# Rationale: pipeline makes the structural one-row-per-country call (newer-wins) and KEEPS methodology_round
# as the discriminator, so the metric pass can still filter/down-weight the newer-round countries for
# comparability. This mirrors the RTI/carbon precedent (make the structural call, flag it, leave it revisable).

# --- 4a. Drop non-assessment rows: a row with no report_date or no report_type is not a usable assessment ---
before = len(fatf_raw)
fatf = fatf_raw[fatf_raw["report_date"].notna() & fatf_raw["report_type"].notna()].copy()
print(f"Dropped {before - len(fatf)} rows with null report_date/report_type (spacer/footnote rows)")

# --- 4b. Rank report_type by completeness, for tie-breaking when a country has same-dated rows ---
# A consolidated row (MER folded with all follow-ups) gives the fullest current rating set; a lone FUR
# may only re-rate a few cells. So on equal dates we prefer the most consolidated type.
# Higher rank = more complete/preferred. Types not listed get rank 0 (least preferred) so nothing is silently dropped.
type_rank = {
    "MER+FUR(s)+FUAR": 5,   # consolidated MER + follow-ups + follow-up assessment
    "MER+FUR(s)":      4,   # consolidated MER + follow-ups  (cleanest "current state")
    "MER":             3,   # original mutual evaluation
    "FUAR":            2,   # follow-up assessment report
    "FUR":             1,   # follow-up report (incremental re-rating)
}
fatf["_type_rank"] = fatf["report_type"].map(type_rank).fillna(0).astype(int)

# Flag any report_type we didn't anticipate (rank 0) so we notice new categories on future refreshes.
_unranked = sorted(set(fatf.loc[fatf["_type_rank"] == 0, "report_type"].unique()))
if _unranked:
    print(f"⚠️  Unranked report_type value(s) present (treated as least-preferred): {_unranked}")

# --- 4c. Within each (jurisdiction, methodology_round): pick latest report_date, tie-break on _type_rank ---
# Sort so the preferred row is last, then keep last per group. report_date is 'YYYY-MM' string → lexicographic
# sort equals chronological, so no date parsing needed (and no hardcoded dates).
fatf_sorted = fatf.sort_values(["jurisdiction", "methodology_round", "report_date", "_type_rank"])
one_per_round = fatf_sorted.groupby(["jurisdiction", "methodology_round"], as_index=False).last()
print(f"After latest-per-(country,round): {len(one_per_round)} rows")

# --- 4d. Collapse across rounds: newer methodology round wins per country ---
# methodology_round is a string token ('2013','2022',...). Lexicographic max = newest round here, but to be
# robust we rank by numeric value parsed from the token rather than assume string order.
one_per_round["_round_num"] = one_per_round["methodology_round"].str.extract(r"(\d+)").astype(int)
one_per_country = (
    one_per_round.sort_values(["jurisdiction", "_round_num"])
    .groupby("jurisdiction", as_index=False).last()
)

# --- 4e. Clean up helper columns ---
fatf_final = one_per_country.drop(columns=["_type_rank", "_round_num"])

print(f"\nFinal: {len(fatf_final)} countries (one row each)")
print("Methodology round distribution:")
print(fatf_final["methodology_round"].value_counts())

Dropped 2 rows with null report_date/report_type (spacer/footnote rows)
After latest-per-(country,round): 206 rows

Final: 199 countries (one row each)
Methodology round distribution:
methodology_round
2013    192
2022      7
Name: count, dtype: int64


In [5]:
# Cell 5 — Dual encoding: keep raw ordinal codes as-is, add parallel numeric columns for the metric pass.
# Two separate ordinal scales. Higher numeric = better in both.
#   Effectiveness (IO1-11):    HE=3 High, SE=2 Substantial, ME=1 Moderate, LE=0 Low
#   Technical compliance (R1-40): C=3 Compliant, LC=2 Largely, PC=1 Partially, NC=0 Non-compliant
# 'N/A' on a Recommendation = "not applicable to this country" (a structural exclusion, NOT a low score),
# so it maps to NaN (dropped from any average), never 0. Blanks/unrecognised codes also -> NaN, and we
# flag any unrecognised code so a future refresh introducing new symbols is caught, not silently nulled.

import numpy as np

io_map  = {"HE": 3, "SE": 2, "ME": 1, "LE": 0}                 # effectiveness scale
rec_map = {"C": 3, "LC": 2, "PC": 1, "NC": 0}                  # technical-compliance scale ('N/A' intentionally absent -> NaN)

io_cols  = [f"IO{i}" for i in range(1, 12)]
rec_cols = [f"R{i}"  for i in range(1, 41)]

def _norm(s):
    # Normalise a rating cell to a clean uppercase token for mapping (strip whitespace/newlines); keep None as None.
    return s.strip().upper() if isinstance(s, str) else s

# Collect any raw codes that fail to map (excluding the expected blanks/N/A) so we notice scale changes.
_unrecognised = set()

def encode(col, mapping):
    raw = fatf_final[col].map(_norm)
    num = raw.map(mapping)                                     # unmapped -> NaN
    # Anything non-null in raw that didn't map, and isn't the known 'N/A', is an unexpected code worth flagging.
    bad = raw[(raw.notna()) & (num.isna()) & (raw != "N/A")]
    _unrecognised.update(bad.unique())
    return num

# Build numeric columns alongside the raw (raw columns are left untouched for traceability).
for c in io_cols:
    fatf_final[f"{c}_num"] = encode(c, io_map)
for c in rec_cols:
    fatf_final[f"{c}_num"] = encode(c, rec_map)

if _unrecognised:
    print(f"⚠️  Unrecognised rating code(s) found (left as NaN — investigate, scale may have changed): {sorted(_unrecognised)}")
else:
    print("All rating codes mapped cleanly (plus expected N/A → NaN).")

# Quick integrity check: numeric ranges should sit within their scale bounds.
io_num_cols  = [f"{c}_num" for c in io_cols]
rec_num_cols = [f"{c}_num" for c in rec_cols]
print(f"\nIO numeric range:  min={fatf_final[io_num_cols].min().min()}, max={fatf_final[io_num_cols].max().max()}  (expect 0-3)")
print(f"R numeric range:   min={fatf_final[rec_num_cols].min().min()}, max={fatf_final[rec_num_cols].max().max()}  (expect 0-3)")
print(f"Columns now: {fatf_final.shape[1]}")

All rating codes mapped cleanly (plus expected N/A → NaN).

IO numeric range:  min=0, max=3  (expect 0-3)
R numeric range:   min=0.0, max=3.0  (expect 0-3)
Columns now: 109


In [9]:
# Cell 6 — Harmonise jurisdiction names to ISO3 (explicit overrides → exact lookup → fuzzy → flag).
# FATF names use curly apostrophes (U+2019) and parenthetical forms pycountry misses, and fuzzy search
# wrongly maps some non-sovereign jurisdictions to their parent state (e.g. Sint Maarten → NLD).
# So we override explicitly for every known mismatch, then FLAG anything unresolved and any duplicate ISO3.
import pycountry

# Overrides keyed to the EXACT strings as they appear in the data (curly apostrophe ’ = U+2019).
ISO3_OVERRIDES = {
    "China (People’s Republic of)":      "CHN",
    "Côte d’Ivoire":                     "CIV",
    "Lao People’s Democratic Republic":  "LAO",
    "Hong Kong (China)":                 "HKG",
    "Macau (China)":                     "MAC",
    "Bailiwick of Guernsey":             "GGY",
    "Sint Maarten":                      "SXM",   # FATF assesses separately; fuzzy wrongly maps to NLD
    "Democratic Republic of the Congo":  "COD",
    "Republic of the Congo":             "COG",   # the other Congo — pre-empt the sibling mismatch   # FATF assesses separately; fuzzy wrongly maps to NLD
    # known-safe extras for future refreshes / alternate spellings:
    "Korea":                             "KOR",
    "Chinese Taipei":                    "TWN",
    "Türkiye":                           "TUR",
}

def to_iso3(name):
    # 1) exact override; 2) override after normalising curly→straight apostrophe; 3) exact lookup;
    # 4) fuzzy search; 5) give up -> None (flagged, never silently dropped).
    if name in ISO3_OVERRIDES:
        return ISO3_OVERRIDES[name]
    norm = name.replace("\u2019", "'")                     # curly ’ -> straight ' to catch apostrophe-only diffs
    if norm in ISO3_OVERRIDES:
        return ISO3_OVERRIDES[norm]
    try:
        return pycountry.countries.lookup(name.strip()).alpha_3
    except LookupError:
        pass
    try:
        return pycountry.countries.search_fuzzy(name.strip())[0].alpha_3
    except (LookupError, Exception):
        return None

fatf_final["iso3"] = fatf_final["jurisdiction"].map(to_iso3)

# Flag unresolved — nothing dropped silently.
unmatched = sorted(fatf_final.loc[fatf_final["iso3"].isna(), "jurisdiction"].unique())
print(f"ISO3 resolved: {fatf_final['iso3'].notna().sum()} / {len(fatf_final)} countries")
if unmatched:
    print(f"\n⚠️  {len(unmatched)} jurisdiction(s) STILL unresolved (add overrides):")
    for j in unmatched:
        print("    ", repr(j))
else:
    print("All jurisdictions resolved to ISO3.")

# Duplicate ISO3 = two jurisdictions collapsed to one country — must be zero before save.
dups = fatf_final.loc[fatf_final["iso3"].notna(), "iso3"].value_counts()
dups = dups[dups > 1]
if len(dups):
    print(f"\n⚠️  Duplicate ISO3 (two jurisdictions → same country): {dups.to_dict()}")
    for code in dups.index:
        print(f"   {code}:", fatf_final.loc[fatf_final['iso3']==code, 'jurisdiction'].tolist())
else:
    print("No duplicate ISO3 codes — every country maps uniquely.")

ISO3 resolved: 199 / 199 countries
All jurisdictions resolved to ISO3.
No duplicate ISO3 codes — every country maps uniquely.


In [12]:
# Cell 7 — Validate, derive as-of date from the data, and save fatf_clean.csv (overwrites existing).

# --- 7a. Assemble final column order: keys first, then raw ratings, then numeric ratings, then meta ---
io_cols   = [f"IO{i}"  for i in range(1, 12)]
rec_cols  = [f"R{i}"   for i in range(1, 41)]
io_num    = [f"{c}_num" for c in io_cols]
rec_num   = [f"{c}_num" for c in rec_cols]
key_cols  = ["iso3", "jurisdiction", "methodology_round", "report_type", "report_date", "assessment_body"]
meta_cols = ["n_upgrades", "n_downgrades"]
ordered   = key_cols + io_cols + rec_cols + io_num + rec_num + meta_cols

# Guard: every expected column must exist (catches an upstream rename before it corrupts the output).
missing = [c for c in ordered if c not in fatf_final.columns]
assert not missing, f"Expected columns missing from fatf_final: {missing}"
fatf_out = fatf_final[ordered].copy()

# --- 7b. Integrity checks — fail loudly rather than save a broken file ---
assert fatf_out["iso3"].notna().all(),            "Null iso3 present — harmonisation incomplete"
assert fatf_out["iso3"].is_unique,                "Duplicate iso3 — one-row-per-country violated"
assert fatf_out["methodology_round"].notna().all(), "Null methodology_round present"
# Numeric ratings must sit in 0–3 (ignoring NaN from N/A); catch any encoding drift.
# Test only non-null values: NaN (from N/A) is a legitimate exclusion, not a range failure.
_num = fatf_out[io_num + rec_num]
_vals = _num.values.ravel()
_vals = _vals[~pd.isna(_vals)]                     # drop NaN before range-testing
assert ((_vals >= 0) & (_vals <= 3)).all(),        "Numeric rating outside 0–3 range"
print(f"Integrity checks passed: {len(fatf_out)} countries, {fatf_out.shape[1]} columns")

# --- 7c. Derive data as-of date FROM THE DATA (max report_date across surviving rows) — never hardcoded ---
data_as_of = fatf_out["report_date"].max()        # 'YYYY-MM' string; lexicographic max = latest
print(f"Data as-of (max report_date in surviving rows): {data_as_of}")
print(f"Methodology rounds present: {sorted(fatf_out['methodology_round'].unique())}")

# --- 7d. Save to processed, overwriting any existing copy ---
out_path = Path(PROCESSED_DIR) / "fatf_clean.csv"
fatf_out.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}  ({out_path.stat().st_size:,} bytes)")

Integrity checks passed: 199 countries, 110 columns
Data as-of (max report_date in surviving rows): 2026-06
Methodology rounds present: ['2013', '2022']

Saved → /Users/boulanger/Documents/governance-framework/data/processed/fatf_clean.csv  (59,455 bytes)


In [13]:
# Cell 8 — Record this build in the download-log and the source-registry (house pattern, mirrors nb32/FARI).
# Manual-download source (Cloudflare-gated), so we log it as a manual snapshot with currency derived from the data.
from datetime import datetime

# Currency is DERIVED (max report_date in the data), never hardcoded. retrieval_date = today.
data_as_of = fatf_out["report_date"].max()                       # 'YYYY-MM' (computed in Cell 7 too; recomputed here so cell is self-contained)
retrieval_date = datetime.today().strftime("%Y-%m-%d")
n_countries = len(fatf_out)
rounds_present = ", ".join(sorted(fatf_out["methodology_round"].unique()))
n_2022 = int((fatf_out["methodology_round"] == "2022").sum())

# --- 8a. Download log ---
update_entry(
    "FATF",
    last_successful_download_date=retrieval_date,
    data_as_of_date=f"{data_as_of} (latest report_date; methodologies {rounds_present})",
    local_filename="fatf_clean.csv",
    latest_available_version=f"Consolidated Assessment Ratings (2013 + 2022 methodologies), as-of {data_as_of}",
    notes=(
        "FATF Mutual Evaluation ratings (AML/CFT regulatory quality, Concept 9). PRIMARY. "
        "MANUAL: fatf-gafi.org Cloudflare-gated; download BOTH consolidated-assessment-ratings-*-methodology.xlsx "
        "by hand (browser) to Downloads; pipeline auto-detects via glob, parses round from filename. "
        "11 Immediate Outcomes (effectiveness HE/SE/ME/LE) + 40 Recommendations (technical compliance C/LC/PC/NC), "
        "raw + numeric (0-3, higher=better; N/A->NaN). One row per country, newer methodology round wins, "
        f"methodology_round flag retained for metric-pass comparability handling. {n_countries} countries "
        f"({n_2022} on 2022 methodology, rest 2013). Cannot auto-fetch (Cloudflare); manual snapshot."
    ),
)
print_entry("FATF")

# --- 8b. Source registry ---
import os
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

fatf_notes = (
    "FATF Consolidated Assessment Ratings (AML/CFT, Concept 9). PRIMARY. MANUAL: site Cloudflare-gated; "
    "download both consolidated-assessment-ratings-2013-methodology.xlsx and -2022-methodology.xlsx by hand "
    "(browser) to Downloads. Pipeline globs version-agnostically, parses methodology round from filename, "
    "skips 3 banner rows, takes first 57 cols, one-row-per-country (latest report_date, consolidated-type "
    "tie-break, newer round wins), encodes IO/R to 0-3 (N/A->NaN), ISO3 via pycountry+overrides. "
    f"{n_countries} countries, methodology_round flag retained. Comparability of 2013 vs 2022 scales "
    "deferred to metric pass (flag enables filter/down-weight)."
)
approach = (
    "manual browser download (Cloudflare-gated) of both methodology .xlsx to Downloads; glob + parse round "
    "from filename, skiprows=3, one-row-per-country newer-round-wins, dual ordinal encoding, ISO3 via pycountry"
)

if (registry_df["source_id"] == "FATF").any():
    registry_df.loc[registry_df["source_id"] == "FATF", "access_method"]   = "manual_download"
    registry_df.loc[registry_df["source_id"] == "FATF", "python_approach"] = approach
    registry_df.loc[registry_df["source_id"] == "FATF", "notes"]           = fatf_notes
    print("Updated FATF registry row")
else:
    registry_df = pd.concat([registry_df, pd.DataFrame([{
        "source_id": "FATF", "access_method": "manual_download",
        "python_approach": approach, "notes": fatf_notes}])], ignore_index=True)
    print("Added FATF registry row")

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print(registry_df[registry_df["source_id"] == "FATF"][["source_id", "access_method"]].to_string(index=False))

[download_log] Updated entry for FATF
  source_id: FATF
  last_attempted_date: 2026-06-26
  last_successful_download_date: 2026-06-26
  data_as_of_date: 2026-06 (latest report_date; methodologies 2013, 2022)
  local_filename: fatf_clean.csv
  latest_available_version: Consolidated Assessment Ratings (2013 + 2022 methodologies), as-of 2026-06
  no_update_reason: nan
  notes: FATF Mutual Evaluation ratings (AML/CFT regulatory quality, Concept 9). PRIMARY. MANUAL: fatf-gafi.org Cloudflare-gated; download BOTH consolidated-assessment-ratings-*-methodology.xlsx by hand (browser) to Downloads; pipeline auto-detects via glob, parses round from filename. 11 Immediate Outcomes (effectiveness HE/SE/ME/LE) + 40 Recommendations (technical compliance C/LC/PC/NC), raw + numeric (0-3, higher=better; N/A->NaN). One row per country, newer methodology round wins, methodology_round flag retained for metric-pass comparability handling. 199 countries (7 on 2022 methodology, rest 2013). Cannot auto-fetch (C

In [14]:
# Cell 9 — Build summary: human-readable readout of the final fatf_clean.csv.
print("=" * 60)
print("FATF MUTUAL EVALUATION RATINGS — BUILD SUMMARY")
print("=" * 60)
print(f"Countries:            {len(fatf_out)}")
print(f"Methodology rounds:   {fatf_out['methodology_round'].value_counts().to_dict()}")
print(f"Data as-of:           {fatf_out['report_date'].max()} (latest report_date)")
print(f"Report date range:    {fatf_out['report_date'].min()} → {fatf_out['report_date'].max()}")
print(f"Columns:              {fatf_out.shape[1]} (6 keys + 11 IO + 40 R raw + 51 numeric + 2 meta)")
print()

# Effectiveness & compliance snapshot using FATF's own summary thresholds (from their State-of-Effectiveness report):
#   TC threshold  = share of 40 Recommendations rated C or LC  (numeric >= 2)
#   Eff threshold = share of 11 Immediate Outcomes rated HE or SE (numeric >= 2)
io_num  = [f"IO{i}_num" for i in range(1, 12)]
rec_num = [f"R{i}_num"  for i in range(1, 41)]

tc_share  = (fatf_out[rec_num] >= 2).sum(axis=1) / fatf_out[rec_num].notna().sum(axis=1)
eff_share = (fatf_out[io_num]  >= 2).sum(axis=1) / fatf_out[io_num].notna().sum(axis=1)

print("Technical compliance (share of Recs rated C/LC), distribution:")
print(tc_share.describe()[["min", "25%", "50%", "75%", "max"]].round(2).to_string())
print("\nEffectiveness (share of IOs rated HE/SE), distribution:")
print(eff_share.describe()[["min", "25%", "50%", "75%", "max"]].round(2).to_string())

# Spot-check a few known countries for face validity (high-capacity should score high, fragile low).
print("\nFace-validity spot check (TC share / Eff share):")
for code in ["GBR", "SGP", "USA", "MMR", "HTI"]:
    row = fatf_out[fatf_out["iso3"] == code]
    if len(row):
        i = row.index[0]
        print(f"  {code} ({row.iloc[0]['jurisdiction']:<20}) round={row.iloc[0]['methodology_round']}  "
              f"TC={tc_share[i]:.2f}  Eff={eff_share[i]:.2f}")
    else:
        print(f"  {code}: not in dataset")

FATF MUTUAL EVALUATION RATINGS — BUILD SUMMARY
Countries:            199
Methodology rounds:   {'2013': 192, '2022': 7}
Data as-of:           2026-06 (latest report_date)
Report date range:    2017-12 → 2026-06
Columns:              110 (6 keys + 11 IO + 40 R raw + 51 numeric + 2 meta)

Technical compliance (share of Recs rated C/LC), distribution:
min    0.11
25%    0.71
50%    0.88
75%    0.92
max    1.00

Effectiveness (share of IOs rated HE/SE), distribution:
min    0.00
25%    0.00
50%    0.09
75%    0.36
max    0.91

Face-validity spot check (TC share / Eff share):
  GBR (United Kingdom      ) round=2013  TC=0.97  Eff=0.73
  SGP (Singapore           ) round=2022  TC=0.95  Eff=0.64
  USA (United States       ) round=2013  TC=0.80  Eff=0.73
  MMR (Myanmar             ) round=2013  TC=0.65  Eff=0.00
  HTI (Haiti               ) round=2013  TC=0.50  Eff=0.00
